# Project 3: a four-vector class and a Higgs processor

In project 2 a four-momentum was a dictionary, and three separate functions worked on it.
Here you write a class that keeps the data and the methods together, and then a processor class that runs the whole analysis on chunks of events.
This is the shape of an analysis in the frameworks you will meet later in the school, such as coffea.

<img src="../img/class-objects.svg" width="80%" alt="">

**How to work**

- Run each cell with `Shift` + `Enter`, from top to bottom.
- Replace every `...` with your own code.
- Each part ends with a `check`. ✅ means move on. ❌ comes with a hint.
- If you are stuck for more than five minutes, ask an instructor.

In [ ]:
import hist
import numpy as np

import sys
sys.path.append("..")  # checks.py lives in the repository root

from checks import check

data = np.load("../data/higgs_eemumu.npz")
fields = ["pt", "eta", "phi", "mass", "charge"]
events = {
    lepton: {field: data[lepton + "_" + field] for field in fields}
    for lepton in ["e1", "e2", "mu1", "mu2"]
}
e1, e2, mu1, mu2 = events["e1"], events["e2"], events["mu1"], events["mu2"]

`events` holds the same 30,868 events as project 2: one dictionary per lepton, one array per field.

## Part 1: `__init__`

`__init__` stores the data on a new object.

The cell below holds the whole class, with three gaps.
Complete one method at a time, run the class cell again, then run that part's check.

**Task.** Complete `__init__` so that it stores all four components.

In [ ]:
class FourVector:
    def __init__(self, E, px, py, pz):
        self.E = E
        self.px = ...
        self.py = ...
        self.pz = ...

    def __add__(self, other):
        return ...

    @property
    def mass(self):
        return ...

In [ ]:
v = FourVector(10.0, 1.0, 2.0, 3.0)
check("3.1", v)

## Part 2: `__add__`

`__add__` makes `+` work.

**Task.** Complete `__add__`: it returns a new `FourVector` whose four components are the sums of the two.

In [ ]:
w = FourVector(10.0, 1.0, 2.0, 3.0) + FourVector(5.0, 0.0, -1.0, 1.0)
check("3.2", w)

## Part 3: Properties

A property is computed when you read it.

**Task.** Complete `mass`: $m = \sqrt{E^2 - p_x^2 - p_y^2 - p_z^2}$, with `np.sqrt` so that it also works on arrays.

In [ ]:
check("3.3", FourVector(5.0, 1.0, 2.0, 2.0).mass)

## Part 4: Leptons as four-vectors

One function turns a lepton into a `FourVector`.

**Task.** Complete `lepton_vector`. The formulas are the ones from projects 1 and 2.

In [ ]:
def lepton_vector(lepton):
    px = lepton["pt"] * np.cos(lepton["phi"])
    py = ...
    pz = ...
    E = ...
    return FourVector(E, px, py, pz)


z_ee = lepton_vector(e1) + lepton_vector(e2)
check("3.4", z_ee.mass)

## Part 5: The processor

A processor holds the settings and the analysis.

`__init__` stores the Z mass windows. `process` runs the analysis of project 2 on some events and returns a dictionary of outputs.

**Task.** Fill in `opposite`, `z_mumu` and what to fill into the histogram.

<img src="../img/process-chunks.svg" width="75%" alt="">

In [ ]:
class HiggsProcessor:
    def __init__(self, z_small=(12, 120), z_big=(40, 120)):
        self.z_small = z_small
        self.z_big = z_big

    def process(self, events):
        e1, e2, mu1, mu2 = events["e1"], events["e2"], events["mu1"], events["mu2"]
        opposite = ...
        z_ee = lepton_vector(e1) + lepton_vector(e2)
        z_mumu = ...
        small = np.minimum(z_ee.mass, z_mumu.mass)
        big = np.maximum(z_ee.mass, z_mumu.mass)
        in_small = (self.z_small[0] < small) & (small < self.z_small[1])
        in_big = (self.z_big[0] < big) & (big < self.z_big[1])
        selected = opposite & in_small & in_big

        h = hist.Hist.new.Reg(50, 80, 180, label="four-lepton mass [GeV]").Double()
        h.fill(...)
        return {"events": len(selected), "selected": int(selected.sum()), "mass": h}


output = HiggsProcessor().process(events)
check("3.5", output["selected"])

In [ ]:
output["mass"].plot();

## Part 6: Adding chunks

Outputs from separate chunks add up.

An analysis framework never gives `process` all the events at once.
It splits them into chunks, calls `process` on each chunk, and adds the outputs.
`split` below cuts `events` into chunks for you.

**Task.** Add each chunk's output to `total`, key by key. Numbers and histograms both add with `+`.

In [ ]:
def split(events, n_chunks):
    """Cut every array in events into n_chunks pieces."""
    edges = np.linspace(0, len(events["e1"]["pt"]), n_chunks + 1).astype(int)
    return [
        {lepton: {field: values[a:b] for field, values in arrays.items()} for lepton, arrays in events.items()}
        for a, b in zip(edges[:-1], edges[1:])
    ]


chunks = split(events, 10)
processor = HiggsProcessor()
total = processor.process(chunks[0])
for chunk in chunks[1:]:
    output = processor.process(chunk)
    total = ...

check("3.6", total["selected"])

In [ ]:
total["mass"].plot();

Ten chunks give the same answer as all events at once.

## Bonus: Inheritance

Inheritance changes only what differs.

**Task.** Write `TightHiggsProcessor`, which inherits from `HiggsProcessor` and only changes the windows: the lighter Z between 20 and 120 GeV, the heavier between 60 and 120 GeV.
Use `super().__init__(...)`.

In [ ]:
class TightHiggsProcessor(HiggsProcessor):
    ...


check("3.7", TightHiggsProcessor().process(events)["selected"])

**Done.** In the analysis frameworks later in the school, such as coffea, the processor you write has the same shape: settings in `__init__`, the analysis in `process(self, events)`.